In [ ]:
import marimo as mo
import plotly.express as px
import polars as pl

from agentnet import (
    DATA_DIR,
    IMAGE_MANIFEST_FILE,
    IMAGES_DIR,
    SUBSET_FILES,
    png_size,
    scan_jsonl,
    task_table,
)

In [ ]:
manifest = pl.read_csv(IMAGE_MANIFEST_FILE)

In [ ]:
mo.md(f"""
# AgentNet (Ubuntu) screenshot exploratory data analysis

**Goal.** Describe the screenshots that accompany the Ubuntu trajectories: resolution, file
size, how they differ by domain, and whether the downloaded sample represents the whole
Ubuntu subset.

**Input.** A seeded, domain stratified sample of tasks, fetched by
`scripts/download_agentnet_images.py`. Dataset revision `{manifest["revision"][0]}`, seed
`{manifest["seed"][0]}`, requested tasks `{manifest["requested_tasks"][0]}`. Every image was
CRC32 verified on download and its sha256 is stored in the manifest.

**Conclusion.** To be written after reviewing the results below.
""")

In [ ]:
import hashlib

unique = manifest.unique("name")
on_disk = [(IMAGES_DIR / name).exists() for name in unique["name"]]
mismatched = sum(
    hashlib.sha256((IMAGES_DIR / name).read_bytes()).hexdigest() != digest
    for name, digest in zip(unique["name"], unique["sha256"])
)
mo.vstack(
    [
        mo.md("## Integrity"),
        mo.ui.table(
            pl.DataFrame(
                {
                    "tasks": [manifest["task_id"].n_unique()],
                    "steps": [manifest.height],
                    "unique_images": [unique.height],
                    "missing_on_disk": [on_disk.count(False)],
                    "sha256_mismatch": [mismatched],
                    "reused_image_names": [manifest.height - unique.height],
                }
            ),
            selection=None,
        ),
    ]
)

## Representativeness of the sample

The sample is stratified by domain, so domain shares should match the full Ubuntu subset.
Trajectory length is not a stratification variable, so it is a genuine check.

In [ ]:
population = task_table(scan_jsonl(DATA_DIR / SUBSET_FILES["ubuntu"])).collect()
sampled_ids = manifest["task_id"].unique()
sample = population.filter(pl.col("task_id").is_in(sampled_ids.implode()))
comparison = (
    population.group_by("domain")
    .agg(pl.len().alias("population_tasks"), pl.col("n_steps").mean().alias("population_steps"))
    .join(
        sample.group_by("domain").agg(
            pl.len().alias("sample_tasks"), pl.col("n_steps").mean().alias("sample_steps")
        ),
        on="domain",
    )
    .with_columns(
        (pl.col("population_tasks") / population.height).alias("population_share"),
        (pl.col("sample_tasks") / sample.height).alias("sample_share"),
    )
    .sort("domain")
)
overall = pl.DataFrame(
    {
        "mean_steps_population": [population["n_steps"].mean()],
        "mean_steps_sample": [sample["n_steps"].mean()],
        "scored_rate_population": [population["task_completed"].is_not_null().mean()],
        "scored_rate_sample": [sample["task_completed"].is_not_null().mean()],
    }
)
mo.vstack(
    [
        mo.ui.table(overall, selection=None),
        mo.ui.table(comparison, selection=None),
    ]
)

In [ ]:
sizes = unique.with_columns(
    pl.Series(
        "dimensions",
        [png_size(IMAGES_DIR / name) for name in unique["name"]],
        dtype=pl.Array(pl.Int64, 2),
    )
).with_columns(
    pl.col("dimensions").arr.get(0).alias("width"),
    pl.col("dimensions").arr.get(1).alias("height"),
    (pl.col("uncompressed_bytes") / 1e6).alias("megabytes"),
)

In [ ]:
mo.vstack(
    [
        mo.md("## Resolutions"),
        mo.ui.table(
            sizes.group_by("width", "height")
            .agg(pl.len().alias("images"), pl.col("megabytes").mean().alias("mean_megabytes"))
            .sort("images", descending=True),
            selection=None,
        ),
    ]
)

In [ ]:
px.histogram(
    sizes,
    x="megabytes",
    nbins=60,
    color_discrete_sequence=["steelblue"],
    title="PNG file size (MB)",
)

In [ ]:
px.box(
    sizes.sort("domain"),
    x="megabytes",
    y="domain",
    color_discrete_sequence=["steelblue"],
    title="PNG file size by domain (MB)",
)

In [ ]:
task_ids = manifest["task_id"].unique().sort().to_list()
task_picker = mo.ui.dropdown(task_ids, value=task_ids[0], label="Task")
task_picker

In [ ]:
trajectory = (
    scan_jsonl(DATA_DIR / SUBSET_FILES["ubuntu"])
    .filter(pl.col("task_id") == task_picker.value)
    .select("instruction", "traj")
    .collect()
    .row(0, named=True)
)
mo.vstack(
    [
        mo.md(f"**Instruction.** {trajectory['instruction']}"),
        *[
            mo.vstack(
                [
                    mo.md(
                        f"Step {step['index']}: {step['value']['action']}  \n"
                        f"`{step['value']['code']}`"
                    ),
                    mo.image(src=IMAGES_DIR / step["image"], width=480),
                ]
            )
            for step in trajectory["traj"][:6]
        ],
    ]
)